# Baseline: import, histograms, segmentation

1. Import the image (config `input`) and save it to `.zarr`.
2. Per-pixel histograms of every channel.
3. Nuclear segmentation (DAPI, always).
4. Second segmentation with another marker (config `segmentation.cells`).
5. Mean, sum and max intensity of every channel per object.

The same steps run without the notebook with `python -m macroseg.baseline <config>`.

In [ ]:
from pathlib import Path

from spatialdata import read_zarr

from macroseg.baseline import measure, segment
from macroseg.config import load_config
from macroseg.histograms import channel_summary, plot_histograms, sample_pixels
from macroseg.ingest import ingest

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = load_config(ROOT / "configs" / "exp201_mouse_neuroblastoma.yaml")

# None = paths from the config (Mewtwo). On the Mac, use the small mouse crop:
INPUT = None  # ROOT / "data/dev/exp201_crop_y6016_x4224_1024.ome.tif"
ZARR = None   # ROOT / "data/dev/exp201_crop.zarr"
OVERWRITE = False

zarr_path = Path(ZARR or cfg.zarr)
element = cfg.image.element

## 1. Import

In [ ]:
if zarr_path.exists() and not OVERWRITE:
    print(f"Reusing {zarr_path}")
else:
    zarr_path.parent.mkdir(parents=True, exist_ok=True)
    ingest(cfg.input, element, INPUT).write(zarr_path, overwrite=OVERWRITE)
sdata = read_zarr(zarr_path)
sdata

## 2. Pixel histograms
Log axes; dashed lines at the 1st, 50th and 99th percentiles. Large images are subsampled on a regular grid.

In [ ]:
pixels = sample_pixels(sdata, element)
plot_histograms(pixels);
channel_summary(pixels)

## 3. Nuclei (DAPI)

In [ ]:
nuclei = cfg.segmentation.nuclei
sdata = segment(sdata, element, nuclei)
sdata = measure(sdata, element, nuclei.labels)
sdata.tables[f"{nuclei.labels}_intensities"].to_df().describe()

## 4. Second segmentation
Channels from `segmentation.cells` in the config (e.g. membrane marker + DAPI).

In [ ]:
cells = cfg.segmentation.cells
if cells is not None:
    sdata = segment(sdata, element, cells)
    sdata = measure(sdata, element, cells.labels)
    print(sdata.tables[f"{cells.labels}_intensities"].obs.filter(regex="^(sum|max)_").describe())
sdata